[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Why PyTorch and TensorFlow


## What you will be able to do

Train a model in PyTorch and in TensorFlow with the library working out every gradient, and check
that its answer matches the one from gradients worked out by hand.

## The idea

### The problem

A weather network has 1,200 stations, and for each one you know its latitude, longitude, elevation,
distance from the coast and mean temperature in January. Fitting a straight line from latitude to
the January mean by gradient descent means starting from a guess, measuring how wrong the line is,
and moving the slope and the intercept a little in the direction that makes it less wrong, a few
hundred times over.

That direction comes from the gradient, the derivative of the error with respect to each parameter.
For a straight line, each of the two derivatives fits on one line of NumPy. Add three more features
and the formulas need matrices. Put a hidden layer of 32 units between the features and the answer
and the model has 193 parameters, each with a derivative that runs through every layer after it.
Working those out by hand, and again whenever the model changes, is where a sign error trains a
model in the wrong direction with no warning.

PyTorch and TensorFlow take that job away. You write how the prediction and the error are computed,
and the library works out every derivative from that code.

### A formal definition

> A **deep learning framework** is a library that provides **tensors**, arrays that can live on a
> CPU or a GPU, and **automatic differentiation**, which computes the gradient of a result with
> respect to every value it was computed from. It also provides the parts a network is built and
> trained with: layers, loss functions and optimizers.

### Why it works that way

As your code runs, the framework records each operation and the values it used: a multiplication,
an addition, a square, a mean. Every operation knows its own derivative, so the framework can walk
the record backward from the error to each parameter, applying the chain rule one operation at a
time. The result is exact to the rounding of the numbers involved, and the walk back costs about as
much as computing the error did, whether the model has two parameters or two million.

The two frameworks keep the record in different places. In PyTorch, a tensor made with
`requires_grad=True` carries its history with it, and calling `backward()` on a result walks that
history. In TensorFlow, operations are recorded while a `tf.GradientTape` is open, and
`tape.gradient` walks the tape. TensorFlow's layers and its training loop come from Keras, a library
installed beside it.

Both frameworks compute in 32-bit floats by default. NumPy uses 64-bit ones, which take twice the
memory for digits a network rarely needs, and mixing the two causes the first of the Common errors.

### Where this shows up

Every notebook in this guide uses both frameworks side by side, PyTorch first. The **scikit-learn,
Deep Dive** guide fitted a linear model to these same made-up stations by solving a formula exactly,
and a network has no such formula, which is why it is trained by gradient descent. The **Automatic
Differentiation** notebook shows what each framework records, and the Visual Handbook covers the
mathematics.

### What this notebook covers

- the stations as NumPy arrays, split into training and test stations
- a line fitted with gradients worked out by hand in NumPy
- the same fit with PyTorch's `backward()` and with TensorFlow's `GradientTape`, compared
- four features, with one line of the loop changed
- the same model built from each framework's parts, then a small network in both
- three errors from these first steps

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

w = torch.tensor(2.0, requires_grad=True)
error = (3.0 * w - 9.0) ** 2
error.backward()
print(error, w.grad)
```

```
tensor(9., grad_fn=<PowBackward0>) tensor(-18.)
```

```python
import tensorflow as tf

w = tf.Variable(2.0)
with tf.GradientTape() as tape:
    error = (3.0 * w - 9.0) ** 2
print(tape.gradient(error, w))
```

```
tf.Tensor(-18.0, shape=(), dtype=float32)
```

The derivative of (3w − 9)² is 2 × (3w − 9) × 3, which is −18 at w = 2. Both frameworks found it
from the code that computed `error`, without being given the formula.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, and the stations.

- `numpy` builds the stations, and fits the first line by hand
- `torch` is PyTorch
- `tensorflow` is TensorFlow, and `keras` its library of layers and training tools
- `contextlib` and `os` hide the messages TensorFlow prints while it loads

TensorFlow's C++ code prints several lines as it loads, each stamped with the time and a process
number, so no two runs would show the same output. Setting `TF_CPP_MIN_LOG_LEVEL` to 3 hides what
TensorFlow logs once it has started, and `c_messages_hidden` sends the lines it prints while it
starts to `os.devnull`. Nothing else in the notebook depends on either. The cell prints PyTorch's
version without the part after `+`, which names the build.

`station_network_arrays` makes the 1,200 weather stations in Norway that the **scikit-learn, Deep
Dive** guide uses. The stations are made up, and a generator started from a fixed seed draws them,
so they are the same on every run. It returns three arrays of 32-bit floats:

- `features`, with a column each for latitude, longitude, elevation in meters and distance from the
  coast in kilometers, in the order `FEATURES` names them
- `jan_mean`, each station's mean temperature in January, in degrees Celsius
- `snow_on_may_1`, 1 for a station with snow on the ground on 1 May and 0 otherwise

The cell then splits the stations at random into 800 for training, 200 for validation and 200 for
testing, and standardizes the features with the training stations' means and standard deviations,
so that every column of `scaled` is centered on 0 with a standard deviation of 1. This notebook uses
the training and test stations, and the **Training Loops** notebook uses the validation stations.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


## Worked examples

### The stations as arrays

In [2]:
print(features.shape, features.dtype)
print(features[:3])
print(jan_mean[:3])
print(len(train_rows), "training stations,", len(test_rows), "test stations")

(1200, 4) float32
[[ 70.93  29.07 358.    17.  ]
 [ 62.97  16.58 311.    11.1 ]
 [ 68.75  21.61  84.    10.  ]]
[-9.6 -3.7 -4.8]
800 training stations, 200 test stations


Each row of `features` is a station. The first one stands at latitude 70.93, longitude 29.07, 358
meters up and 17 kilometers from the coast, and its January mean is −9.6 degrees. `train_rows` and
`test_rows` hold row numbers, so `features[train_rows]` is the training stations.

### A line fitted with gradients worked out by hand

The line predicts the January mean as `w * x + b`, where `x` is a station's latitude, standardized.
How wrong it is, its **loss**, is the mean of the squared errors. Its derivatives with respect to
the slope and the intercept are

- for `w`: 2 × the mean of (`w * x + b − y`) × `x`
- for `b`: 2 × the mean of (`w * x + b − y`)

Gradient descent moves each parameter a step against its derivative, here 0.1 times it, which is
the **learning rate**, and repeats.

In [3]:
x = scaled[train_rows, 0]
y = jan_mean[train_rows]

w, b = 0.0, 0.0
for step in range(101):
    error = w * x + b - y
    loss = (error ** 2).mean()
    if step % 25 == 0:
        print(step, round(float(loss), 3), round(float(w), 3), round(float(b), 3))
    dw = 2 * (error * x).mean()
    db = 2 * error.mean()
    w -= 0.1 * dw
    b -= 0.1 * db

w_numpy, b_numpy = w, b
print("standard deviation of latitude:", round(float(std[0]), 2))
print("degrees per degree of latitude:", round(float(w_numpy / std[0]), 2))

0 34.569 0.0 0.0
25 4.542 -1.977 -5.088
50 4.542 -1.985 -5.108
75 4.542 -1.985 -5.108
100 4.542 -1.985 -5.108
standard deviation of latitude: 3.7
degrees per degree of latitude: -0.54


The loss starts at 34.569, the mean square of the January means themselves, since a line with a
slope and intercept of 0 predicts 0 everywhere. By step 25 it has fallen to 4.542, and it stays
there, because the line has reached the best fit that latitude alone allows. The slope is per standard deviation
of latitude, which is 3.7 degrees, so divided by that it says the January mean falls about 0.54
degrees for every degree north.

### PyTorch works out the same line's gradients

The loop below makes the same prediction and the same loss, from tensors instead of arrays. The two
derivative lines are gone, replaced by `loss.backward()`, which fills `w.grad` and `b.grad`.

In [4]:
x_torch = torch.from_numpy(x)
y_torch = torch.from_numpy(y)

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
for step in range(101):
    loss = ((w * x_torch + b - y_torch) ** 2).mean()
    if step % 25 == 0:
        print(step, round(loss.item(), 3), round(w.item(), 3), round(b.item(), 3))
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    w.grad.zero_()
    b.grad.zero_()

w_torch, b_torch = w, b

0 34.569 0.0 0.0
25 4.542 -1.977 -5.088
50 4.542 -1.985 -5.108
75 4.542 -1.985 -5.108
100 4.542 -1.985 -5.108


Every line of output matches the NumPy loop's. The loop adds three things:

- `requires_grad=True` tells PyTorch to record what is computed from `w` and `b`
- the update runs inside `torch.no_grad()`, because a step that changes the weights is not part of
  the model and must not be recorded
- `w.grad.zero_()` empties the gradient after each step, because `backward()` adds to whatever
  `.grad` already holds, which the **Automatic Differentiation** notebook shows

In [5]:
print(float(w_numpy), w_torch.item())
print(float(b_numpy), b_torch.item())

-1.9848098754882812 -1.9848098754882812
-5.107630252838135 -5.107630252838135


The NumPy loop and the PyTorch loop end at the same 32-bit numbers. Both are printed as Python
floats, which show more digits than a 32-bit float holds, and every digit matches.

### The same line in TensorFlow with GradientTape

In TensorFlow, a value that training changes is a `tf.Variable`, and the loss is computed inside a
`with tf.GradientTape()` block so that the tape records it. `tape.gradient` returns the derivatives,
and `assign_sub` subtracts a step from each variable.

In [6]:
x_tf = tf.constant(x)
y_tf = tf.constant(y)

w = tf.Variable(0.0)
b = tf.Variable(0.0)
for step in range(101):
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean((w * x_tf + b - y_tf) ** 2)
    if step % 25 == 0:
        print(step, round(float(loss), 3), round(float(w), 3), round(float(b), 3))
    dw, db = tape.gradient(loss, [w, b])
    w.assign_sub(0.1 * dw)
    b.assign_sub(0.1 * db)

0 34.569 0.0 0.0
25 4.542 -1.977 -5.088
50 4.542 -1.985 -5.108
75 4.542 -1.985 -5.108
100 4.542 -1.985 -5.108


TensorFlow prints the same numbers again. The tape is made new for every step, because a tape that has given its
gradients once is finished.

### The gradient checked against the formula

The idea said that automatic differentiation is exact. At `w = 0` and `b = 0`, the cell below
prints both derivatives twice, once from the NumPy loop's formulas and once from PyTorch's
`backward()`.

In [7]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
loss = ((w * x_torch + b - y_torch) ** 2).mean()
loss.backward()

print("by hand:", float(2 * ((0 * x + 0 - y) * x).mean()), float(2 * (0 * x + 0 - y).mean()))
print("PyTorch:", w.grad.item(), b.grad.item())

by hand: 3.969588041305542 10.215250015258789
PyTorch: 3.969587802886963 10.215250015258789


The two agree to the seventh significant digit, which is about as far as a 32-bit float carries
one. NumPy and PyTorch add up the 800 terms of each mean in their own order, and rounding depends on
the order, so the digits after that differ. Automatic differentiation computes the derivative
itself, from the record of the operations, with no estimate from nearby points involved.

### All four features with one line of the loop changed

Latitude explains part of the January mean, and elevation and the coast explain more. With all four
features, the hand-written derivatives would have to become matrix formulas. In PyTorch, `w` becomes
four weights and `w * x` becomes `x @ w`, a matrix product of each station's four features with the
four weights. Nothing else in the loop changes.

In [8]:
X_train = torch.from_numpy(scaled[train_rows])
X_test = torch.from_numpy(scaled[test_rows])
y_test = torch.from_numpy(jan_mean[test_rows])

w = torch.zeros(4, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
for step in range(500):
    loss = ((X_train @ w + b - y_torch) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    w.grad.zero_()
    b.grad.zero_()

with torch.no_grad():
    test_loss = ((X_test @ w + b - y_test) ** 2).mean()
    line_test_loss = ((torch.from_numpy(scaled[test_rows, 0]) * w_torch + b_torch - y_test) ** 2).mean()
print("training loss", round(loss.item(), 3), "| test loss", round(test_loss.item(), 3))
print("latitude alone, test loss", round(line_test_loss.item(), 3))
print("weights", w.detach().numpy().round(2))

training loss 0.992 | test loss 1.181
latitude alone, test loss 4.51
weights [-1.87 -0.17 -1.25 -1.05]


The **test loss** is the loss on the 200 stations the loop never saw, which is the fair measure of
a model. Latitude alone leaves 4.51 on them, and the four features 1.181. Elevation and the distance
from the coast get weights of −1.25 and −1.05, close to latitude's −1.87, so per standard deviation
each matters nearly as much as latitude does.

### The same model from each framework's parts

Writing the weights and the update by hand is how the libraries work underneath, and it is rarely
how they are used. In PyTorch, `nn.Linear(4, 1)` holds the four weights and the intercept, which it
calls the **bias**. `nn.MSELoss` computes the mean squared error, and an optimizer from `torch.optim`
makes the step. The loop shrinks to three calls: `zero_grad()`, `backward()` and `step()`.

In [9]:
torch.manual_seed(0)
linear = torch.nn.Linear(4, 1)
optimizer = torch.optim.SGD(linear.parameters(), lr=0.1)
loss_fn = torch.nn.MSELoss()
y_train = y_torch.unsqueeze(1)

for step in range(500):
    optimizer.zero_grad()
    loss = loss_fn(linear(X_train), y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    print("training loss", round(loss.item(), 3), "| test loss",
          round(loss_fn(linear(X_test), y_test.unsqueeze(1)).item(), 3))

training loss 0.992 | test loss 1.181


`nn.Linear` starts from random weights rather than zeros, so `torch.manual_seed(0)` fixes them, and
the model makes one prediction per station in a column, shape (800, 1), which is why the target gets
the same shape from `unsqueeze(1)`. The **Losses and Optimizers** notebook shows what happens
without it.

Keras goes one step further and runs the loop itself. A `Sequential` model with a `Dense` layer of
one unit is the same linear model, `compile` chooses the optimizer and the loss, and `fit` trains.
`batch_size=800` makes every step use all 800 training stations, as the loops above do, and
`verbose=0` stops `fit` printing a progress bar.

In [10]:
keras.utils.set_random_seed(0)
linear_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(1)])
linear_keras.compile(optimizer=keras.optimizers.SGD(0.1), loss="mse")
history = linear_keras.fit(scaled[train_rows], jan_mean[train_rows], epochs=500, batch_size=800, verbose=0)

print("training loss", round(history.history["loss"][-1], 3), "| test loss",
      round(linear_keras.evaluate(scaled[test_rows], jan_mean[test_rows], verbose=0), 3))

training loss 0.992 | test loss 1.181


All three print a training loss of 0.992 and a test loss of 1.181. After 500 steps, each version has
reached the same best line, whatever weights it started from.

### A small network

A linear model can only add up its features, each with a fixed weight. A network puts a layer of
**hidden units** between the features and the answer. Each unit takes its own weighted sum of the
four features, `nn.ReLU` replaces any negative sum with 0, and a last layer adds up the units. That
bend at 0 is what lets the network fit a curve, such as the effect of the coast in these made-up
stations, which fades with distance. Nobody would write this model's 193 derivatives by hand, and
the loop is the same three calls as before. `Adam` is an optimizer that adapts its step for each weight, which the **Losses and
Optimizers** notebook compares with plain gradient descent.

In [11]:
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
print(sum(p.numel() for p in net.parameters()), "parameters")

for step in range(1, 501):
    optimizer.zero_grad()
    loss = loss_fn(net(X_train), y_train)
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        with torch.no_grad():
            test_loss = loss_fn(net(X_test), y_test.unsqueeze(1))
        print(step, "training loss", round(loss.item(), 3), "| test loss", round(test_loss.item(), 3))

193 parameters
100 training loss 1.256 | test loss 1.357
200 training loss 0.89 | test loss 0.99
300 training loss 0.816 | test loss 0.987
400 training loss 0.789 | test loss 0.972
500 training loss 0.782 | test loss 0.96


The training loss falls to 0.782 and the test loss to 0.96, below the linear model's 1.181, so the
network has found something in the stations that a weighted sum of the four features misses.

In [12]:
keras.utils.set_random_seed(0)
net_keras = keras.Sequential([
    keras.Input((4,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1),
])
net_keras.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
history = net_keras.fit(scaled[train_rows], jan_mean[train_rows], epochs=500, batch_size=800, verbose=0)

print(net_keras.count_params(), "parameters")
print("training loss", round(history.history["loss"][-1], 3), "| test loss",
      round(net_keras.evaluate(scaled[test_rows], jan_mean[test_rows], verbose=0), 3))

193 parameters
training loss 0.779 | test loss 0.978


The Keras network has the same 193 parameters and reaches a test loss of 0.978, against PyTorch's
0.96. The two frameworks start from different random weights, so the losses differ in the second
decimal place.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/01-why-pytorch-and-tensorflow-solutions.ipynb).

**1.** Run the NumPy line fit again with a learning rate of 0.01 in place of 0.1, and print the loss every 25 steps to see whether 100 steps is enough.

In [13]:
# your code here

**2.** Fit the January mean to elevation alone, column 2 of `scaled`, with the PyTorch loop, and print the slope in degrees per meter, the slope divided by `std[2]`.

In [14]:
# your code here

**3.** Write the four-feature fit in TensorFlow, with `tf.Variable(tf.zeros(4))` for the weights and `tf.linalg.matvec` for the product, and print the loss on the test stations after 500 steps.

In [15]:
# your code here

**4.** Train the PyTorch network with 64 hidden units in place of 32, from `torch.manual_seed(0)`, and print its loss on the test stations after 500 steps.

In [16]:
# your code here

**5.** Predict the January mean at a planned station at latitude 70.0, longitude 23.0, 300 meters up and 15 kilometers from the coast, with `net` and with `net_keras`.

In [17]:
# your code here

## Common errors

### `RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float`

A station is planned at latitude 69.5, longitude 19.0, 120 meters up and 4 kilometers from the
coast. Its features are standardized with the training stations' means, and given to the network.

In [18]:
planned = np.array([[69.5, 19.0, 120.0, 4.0]])
planned_scaled = (planned - mean) / std
print(planned.dtype, planned_scaled.dtype)
net(torch.from_numpy(planned_scaled))

float64 float64


RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float

`np.array` makes 64-bit floats, which PyTorch calls `Double`, and the network's weights are 32-bit,
`Float`. `mat1` is the input and `mat2` the layer's weights. A matrix product needs both of one type,
and PyTorch does not convert either of them for you. Make the array 32-bit before it becomes a tensor.

In [19]:
with torch.no_grad():
    print(net(torch.from_numpy(planned_scaled.astype("float32"))))

tensor([[-5.7348]])


The network predicts a January mean of −5.7 degrees for the planned station.

### `RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn`

In [20]:
w = torch.tensor(0.0)
b = torch.tensor(0.0)
loss = ((w * x_torch + b - y_torch) ** 2).mean()
loss.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

A tensor made without `requires_grad=True` has no history, so neither does anything computed from it,
and `backward()` has nothing to walk. `grad_fn` is the step in the record that made a tensor, and a
loss computed only from such tensors has none. Make the weights with `requires_grad=True`, as the
worked examples do.

### No error, and a gradient of `None`

The TensorFlow version of the same mistake raises nothing.

In [21]:
w = tf.constant(0.0)
b = tf.constant(0.0)
with tf.GradientTape() as tape:
    loss = tf.reduce_mean((w * x_tf + b - y_tf) ** 2)
print(tape.gradient(loss, [w, b]))

[None, None]


A tape records what is computed from a `tf.Variable` and ignores a constant unless told to watch it,
so both gradients are `None`. Make the weights `tf.Variable`s, as the worked examples do, or call
`tape.watch` inside the block.

In [22]:
w = tf.constant(0.0)
with tf.GradientTape() as tape:
    tape.watch(w)
    loss = tf.reduce_mean((w * x_tf - y_tf) ** 2)
print(tape.gradient(loss, w))

tf.Tensor(3.9695878, shape=(), dtype=float32)


Watched, the constant gets a gradient of 3.9696, the value the gradient check found for `w` at 0.

## Recap

- You can compute a gradient by hand for a line, and have `backward()` or `GradientTape` compute
  it for you.
- You can train a model with a loop of loss, gradient and step in PyTorch and in TensorFlow.
- You can build the same model from `nn.Linear` and an optimizer, and from Keras's `Dense` and `fit`.
- You can turn NumPy's 64-bit arrays into the 32-bit tensors both frameworks expect.

## What is next

The **Tensors** notebook looks at the arrays both frameworks compute with: how they are made, the
dtypes they choose, and how they pass to and from NumPy.

---

[PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Tensors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/02-tensors.ipynb) &#8594;
